# 02 — Emotion word list and prompts

**Goal of this notebook.** Lock in *what* we will generate before *how*. Three things land here, each lifted verbatim from Anthropic (*transformer-circuits.pub/2026/emotions*):

1. A **generator model** to write the emotion-conditioned passages — chosen with eyes open about quality, licence, and Colab fit.
2. The **canonical 171-emotion vocabulary** plus the **100 topic seeds** the paper uses to anchor every story.
3. The **three system prompts** used in the paper:
   - *Emotional stories* — paired with `(emotion, topic)`.
   - *Neutral dialogues* — Person ↔ AI Q&A, the **baseline contrast** (this is what differs from a naive replication: the baseline is dialogues, not neutral stories).
   - *Emotional dialogues* — separate `person_emotion` and `ai_emotion`, used in nb08 for the hidden-vs-expressed probe benchmark.

The notebook also computes a **wall-clock estimate** for the full generation pass on Colab T4 / L4 / A100 vs. local hardware, and emits a frozen JSON spec that nb03 consumes.

**Source-of-truth note.** The verbatim list, topics, and prompt templates also live at `../../EmotionVectorsReview/canonical_emotions_and_prompts.md`. If anything below disagrees with that note, the note wins.

## 1 — Setup

In [ ]:
# Self-installing on Colab; no-op locally.
import importlib, subprocess, sys
def _ensure(pkg, import_name=None):
    try: importlib.import_module(import_name or pkg)
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
for pkg, imp in [("transformers", "transformers"), ("matplotlib", "matplotlib"), ("numpy", "numpy")]:
    _ensure(pkg, imp)


In [ ]:
import json, os, sys, time, math, hashlib, textwrap
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

try:
    from google.colab import drive  # noqa
    IS_COLAB = True
    WORK_DIR = Path("/content/EmoVecLLM"); WORK_DIR.mkdir(exist_ok=True)
except Exception:
    IS_COLAB = False
    WORK_DIR = Path("..").resolve()  # repo root when notebook is in notebooks/

DATA_DIR = WORK_DIR / "data" / "processed"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"IS_COLAB={IS_COLAB}  |  WORK_DIR={WORK_DIR}  |  DATA_DIR={DATA_DIR}")


## 2 — Choose a generator model

Anthropic uses **Claude Sonnet 4.5** to write the stories. We can't (it's closed), so we pick from open-weight models that fit the constraints below.

**What the generator must do.**
- Follow an instruction reliably (chat / instruct-tuned, not a base LM).
- Produce ~200–300-word coherent character vignettes per emotion.
- Be small enough to live alongside the *target* model in VRAM, **or** be run in a separate session and the stories cached to disk.

| Model | Params | T4 (free Colab) | L4 / A100 (Pro) | Licence | Notes |
|---|---|---|---|---|---|
| `gpt2` | 124 M | fp16 ✅ | fp16 ✅ | MIT | Too small for instruction-following — base only, **not viable** as generator. |
| `EleutherAI/pythia-1.4b` | 1.4 B | fp16 ✅ | fp16 ✅ | Apache-2.0 | Base model, no instruction tuning. Usable for prototyping with few-shot prompting; expect uneven story quality. |
| `Qwen/Qwen2.5-7B-Instruct` | 7 B | 4-bit ✅ | fp16 ✅ | Tongyi Qianwen | Strong instruction following; multilingual; permissive but not Apache. |
| `meta-llama/Llama-3.1-8B-Instruct` | 8 B | 4-bit ✅ | fp16 ✅ | Llama-3 (gated) | Headline open generator; needs HF token + license accept. |
| `mistralai/Mistral-7B-Instruct-v0.3` | 7 B | 4-bit ✅ | fp16 ✅ | Apache-2.0 | Drop-in alternative when Llama gating is inconvenient. |

**Default.** `Qwen2.5-7B-Instruct` — Apache-permissive, strong, no gating — for ease of replication.

In [ ]:
# ── Generator model + decoding knobs ──────────────────────────────────────
GENERATOR_MODEL = "Qwen/Qwen2.5-7B-Instruct"

# Generation budget — tunable. Each *call* generates n_stories at once (per
# Anthropic's prompt template that asks for n_stories in a single request).
N_TOPICS_PER_EMOTION   = 3      # of 100; sampled per emotion for emotion stories
N_STORIES_PER_CALL     = 5      # n_stories in the {n_stories} template var
N_NEUTRAL_PER_CALL     = 5      # per-call dialogues for the neutral baseline
TARGET_TOKENS_PER_CALL = 1400   # ≈5 stories × ~250 tokens (one big completion)

# Optional, for nb08 (hidden-vs-expressed). Off by default — adds ~1× emotion budget.
GENERATE_EMOTIONAL_DIALOGUES = False
N_EMOTION_DIALOGUE_PAIRS     = 200   # sampled (person_emotion, ai_emotion) pairs

# Decoding params
TEMPERATURE = 0.9
TOP_P       = 0.95
SEED_BASE   = 0xE3F0   # +i*hash(...) per call for reproducibility

# Hardware profile (for the time estimate; pick what you'll actually run on).
HW_PROFILE = "T4_4bit"  # one of: T4_4bit, T4_fp16, L4_fp16, A100_fp16, RTX4090, CPU

print(f"Generator: {GENERATOR_MODEL}")
print(f"Per emotion: {N_TOPICS_PER_EMOTION} topics × {N_STORIES_PER_CALL} stories/call")
print(f"Neutral baseline: 100 topics × {N_NEUTRAL_PER_CALL} dialogues/call")
print(f"HW profile for time estimate: {HW_PROFILE}")


## 3 — The 171-emotion vocabulary

Anthropic seed their pipeline with a curated list of **171 emotion concepts** spanning valence × arousal and including non-basic granular categories (*brooding*, *infatuated*, *valiant*, *droopy*, *dumbstruck*).

The list is **published verbatim** in the paper's appendix ("Full list of emotions"). We use it directly. It includes a few entries that read more as *traits or states* than canonical emotions (*alert*, *kind*, *patient*, *dependent*, *stuck*, *reflective*) — Anthropic appear to use a permissive *affective concept* definition rather than basic-emotion-theory criteria.

The fitted V/A regression onto our derived vectors happens in nb07 against the actual Warriner et al. 2013 norms — we don't pre-tag here.

In [ ]:
# Canonical 171-emotion list from Anthropic, transformer-circuits.pub/2026/emotions
# §"Full list of emotions" — verbatim. See ../../EmotionVectorsReview/canonical_emotions_and_prompts.md
EMOTIONS_171 = [
    'afraid', 'alarmed', 'alert', 'amazed', 'amused', 'angry',
    'annoyed', 'anxious', 'aroused', 'ashamed', 'astonished', 'at ease',
    'awestruck', 'bewildered', 'bitter', 'blissful', 'bored', 'brooding',
    'calm', 'cheerful', 'compassionate', 'contemptuous', 'content', 'defiant',
    'delighted', 'dependent', 'depressed', 'desperate', 'disdainful', 'disgusted',
    'disoriented', 'dispirited', 'distressed', 'disturbed', 'docile', 'droopy',
    'dumbstruck', 'eager', 'ecstatic', 'elated', 'embarrassed', 'empathetic',
    'energized', 'enraged', 'enthusiastic', 'envious', 'euphoric', 'exasperated',
    'excited', 'exuberant', 'frightened', 'frustrated', 'fulfilled', 'furious',
    'gloomy', 'grateful', 'greedy', 'grief-stricken', 'grumpy', 'guilty',
    'happy', 'hateful', 'heartbroken', 'hope', 'hopeful', 'horrified',
    'hostile', 'humiliated', 'hurt', 'hysterical', 'impatient', 'indifferent',
    'indignant', 'infatuated', 'inspired', 'insulted', 'invigorated', 'irate',
    'irritated', 'jealous', 'joyful', 'jubilant', 'kind', 'lazy',
    'listless', 'lonely', 'loving', 'mad', 'melancholy', 'miserable',
    'mortified', 'mystified', 'nervous', 'nostalgic', 'obstinate', 'offended',
    'on edge', 'optimistic', 'outraged', 'overwhelmed', 'panicked', 'paranoid',
    'patient', 'peaceful', 'perplexed', 'playful', 'pleased', 'proud',
    'puzzled', 'rattled', 'reflective', 'refreshed', 'regretful', 'rejuvenated',
    'relaxed', 'relieved', 'remorseful', 'resentful', 'resigned', 'restless',
    'sad', 'safe', 'satisfied', 'scared', 'scornful', 'self-confident',
    'self-conscious', 'self-critical', 'sensitive', 'sentimental', 'serene', 'shaken',
    'shocked', 'skeptical', 'sleepy', 'sluggish', 'smug', 'sorry',
    'spiteful', 'stimulated', 'stressed', 'stubborn', 'stuck', 'sullen',
    'surprised', 'suspicious', 'sympathetic', 'tense', 'terrified', 'thankful',
    'thrilled', 'tired', 'tormented', 'trapped', 'triumphant', 'troubled',
    'uneasy', 'unhappy', 'unnerved', 'unsettled', 'upset', 'valiant',
    'vengeful', 'vibrant', 'vigilant', 'vindictive', 'vulnerable', 'weary',
    'worn out', 'worried', 'worthless',
]
assert len(EMOTIONS_171) == 171
assert len(set(EMOTIONS_171)) == 171, "duplicate emotion!"
print(f"Loaded {len(EMOTIONS_171)} emotions (Anthropic canonical list).")

multiword = [w for w in EMOTIONS_171 if " " in w or "-" in w]
print(f"multi-word / hyphenated entries ({len(multiword)}): {multiword}")


### 3a — Survey the list

A quick visual of length and first-letter distribution. Purely informational — we trust Anthropic's curation.

In [ ]:
from collections import Counter
import string

lengths = [len(w) for w in EMOTIONS_171]
first_letters = Counter(w[0] for w in EMOTIONS_171)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), constrained_layout=True)
axes[0].hist(lengths, bins=range(2, max(lengths)+2), color="#34618d", edgecolor="white")
axes[0].set_xlabel("word length (characters)"); axes[0].set_ylabel("count")
axes[0].set_title("Length distribution"); axes[0].spines[["top","right"]].set_visible(False)

letters_sorted = sorted(first_letters.keys())
axes[1].bar(letters_sorted, [first_letters[c] for c in letters_sorted], color="#3a7d44")
axes[1].set_xlabel("first letter"); axes[1].set_ylabel("count")
axes[1].set_title("Distribution by first letter"); axes[1].spines[["top","right"]].set_visible(False)
fig.suptitle("171 canonical emotions — overview", fontsize=11)
plt.show()

print()
print("─" * 70)
for c in string.ascii_lowercase:
    words = [w for w in EMOTIONS_171 if w[0] == c]
    if words:
        print(f"  {c}: {', '.join(words)}")


## 4 — The 100 topic seeds

The unit of generation is **`(emotion × topic)`**, not emotion alone. Anthropic anchor every story in one of 100 mundane-but-charged life situations — the kind that pull strong emotional responses without being melodramatic. Flat scenarios make the *emotion-conditioning* carry the variance; hyper-dramatic scenarios would let the *event* drive activations independently of the target emotion.

The same 100 topics also seed the **neutral dialogues** baseline (one Person↔AI Q&A per topic).

In [ ]:
TOPICS_100 = [
    "An artist discovers someone has tattooed their work",
    "A family member announces they're converting to a different religion",
    "Someone's childhood imaginary friend appears in their niece's drawings",
    "A person finds out their biography was written without their knowledge",
    "A neighbor starts a renovation project",
    "Someone finds their grandmother's engagement ring in a pawn shop",
    "A student learns their scholarship application was denied",
    "A person's online friend turns out to live in the same city",
    "A neighbor wants to install a fence",
    "An adult child moves back in with their parents",
    "An employee is asked to train their replacement",
    "An athlete is asked to switch positions",
    "A traveler's flight is delayed, causing them to miss an important event",
    "A student is accused of plagiarism",
    "A person discovers their mentor has retired without saying goodbye",
    "Two friends both apply for the same job",
    "A person runs into their ex at a mutual friend's wedding",
    "Someone discovers their friend has been lying about their job",
    "A person discovers their partner has been taking secret phone calls",
    "A person discovers their child has the same teacher they had",
    "A person's car is towed from their own driveway",
    "Two friends realize they remember a shared event completely differently",
    "Someone discovers their mother kept every school assignment",
    "A person discovers their teenage diary has been published online",
    "Someone finds out their medical records were mixed up with another patient's",
    "A person finds out their article was published under someone else's name",
    "An athlete doesn't make the team they expected to join",
    "An employee is transferred to a different department",
    "Someone receives a friend request from a childhood bully",
    "A person finds out their surprise party has been cancelled",
    "An employee finds out a junior colleague makes more money",
    "A person finds out their partner has been learning their native language",
    "A chef receives a harsh review from a food critic",
    "A person learns their favorite restaurant is closing",
    "Someone finds their childhood teddy bear at a yard sale",
    "A homeowner discovers previous residents left items in the attic",
    "Someone finds an unsigned birthday card in their mailbox",
    "Someone discovers a hidden room in their new house",
    "Two strangers realize they've been dating the same person",
    "A person finds a hidden letter in a used book",
    "Two siblings inherit their grandmother's house",
    "Someone finds a wallet containing a large sum of cash",
    "Someone receives an invitation to their high school reunion",
    "Someone discovers their recipe has become famous under another name",
    "A college student discovers their roommate has been reading their journal",
    "A person finds out they were adopted through a DNA test",
    "A family member wants to sell a cherished heirloom",
    "Someone receives a package intended for the previous tenant",
    "Someone's childhood home is about to be demolished",
    "A person's invention is already patented by someone else",
    "A neighbor's dog keeps escaping into their yard",
    "A coach has to cut a player from the team",
    "Someone learns their favorite author plagiarized their stories",
    "A student finds out their scholarship was meant for someone else",
    "Someone discovers their teenager has a secret social media account",
    "Two roommates disagree about getting a pet",
    "Two friends plan separate birthday parties on the same day",
    "A person learns their childhood best friend doesn't remember them",
    "A musician hears their song being performed by someone else",
    "A person's manuscript is rejected by their dream publisher",
    "A person finds old photos that contradict family stories",
    "A person is asked to give a speech at their parent's retirement party",
    "A student discovers their teacher follows them on social media",
    "A parent finds an old letter they wrote but never sent",
    "An employee discovers the company is being sold",
    "A person accidentally sends a text to the wrong recipient",
    "Two coworkers are stuck in an elevator for three hours",
    "A student learns their thesis advisor is leaving the university",
    "A person's longtime hobby becomes their child's obsession",
    "Two colleagues are both considered for the same promotion",
    "Two coworkers discover they went to the same summer camp",
    "A tenant receives an eviction notice",
    "Someone finds their parent's draft letter of resignation from decades ago",
    "Someone finds out their best friend is moving across the country",
    "A neighbor's tree falls on their property",
    "Someone receives an apology letter years after the incident",
    "A person discovers the tree they planted as a child has been cut down",
    "Two siblings discover different versions of their inheritance",
    "A person finds their childhood home listed for sale online",
    "A homeowner learns their house was a former crime scene",
    "Someone finds out they have a half-sibling they never knew about",
    "A person learns their childhood bully became a therapist",
    "Two people discover they've been working on identical projects",
    "A person finds their spouse's secret savings account",
    "A neighbor complains about noise levels",
    "Someone finds their deceased parent's bucket list",
    "A teacher receives an unexpected gift from a former student",
    "An artist's work is displayed without their permission",
    "Someone discovers their neighbor is secretly wealthy",
    "A student receives a much lower grade than expected",
    "A person learns their college is closing down",
    "A neighbor asks to cut down a tree on the property line",
    "Two strangers discover they share the same rare medical condition",
    "Someone receives flowers with no card attached",
    "Someone discovers their partner has been writing a novel about them",
    "Someone finds a time capsule they don't remember burying",
    "Someone finds their partner's bucket list",
    "A neighbor asks to use part of the yard for a garden",
    "A person learns their apartment building is going condo",
    "Someone finds their college application essay published as an example",
]
assert len(TOPICS_100) == 100
print(f"Loaded {len(TOPICS_100)} topic seeds (Anthropic canonical list).")
print()
for i, t in enumerate(TOPICS_100[:6], 1):
    print(f"  {i:>3d}. {t}")
print("  ...")
for i, t in enumerate(TOPICS_100[-3:], len(TOPICS_100) - 2):
    print(f"  {i:>3d}. {t}")


## 5 — Prompt templates

Three system prompts, all lifted **verbatim** from Anthropic. They are *system* prompts (not user messages): the entire filled template goes into the chat-format `system` slot, and we trigger generation with a minimal user message (`"Begin."`). This is faithful to the paper.

### Why the baseline is dialogues, not stories

If the baseline were neutral *stories* (matched form to the emotion stories), the resulting vector would be `emotion_story − neutral_story` — clean. But that's not what the paper does. Anthropic's neutral baseline is **Person ↔ AI dialogues** — a deliberately different *form* that captures all the things a chat model does *outside* an emotionally-coloured narrative.

The contrast `emotion_story − neutral_dialogue` therefore mixes two things:
1. The actual emotion content.
2. A *story-vs-dialogue* style direction (narrative prose vs. Q&A formatting).

That contamination is real, and Anthropic correct for it explicitly. They take the top principal components of the *neutral dialogue* activations (enough to explain 50 % of the variance) and **project those components out** of the emotion vectors before downstream use. The PCA-projection step happens in nb05 — flagged here so you don't forget it lives downstream.

Why this is the right choice: dialogues are the model's *native operating regime* (instruct models live in chat-formatted Q&A). Subtracting that gives a vector that lives in the *deviation from baseline conversation* — which is exactly the locus where steering interventions act.

### 5a — Emotional stories prompt

In [ ]:
STORY_SYSTEM_PROMPT = """\
Write {n_stories} different stories based on the following premise.

Topic: {topic}

The story should follow a character who is feeling {emotion}.

Format the stories like so:
[story 1]
[story 2]
[story 3]
etc.

The paragraphs should each be a fresh start, with no continuity. Try to make them diverse and not use the same turns of phrase. Across the different stories, use a mix of third-person narration and first-person narration.

IMPORTANT: You must NEVER use the word '{emotion}' or any direct synonyms of it in the stories. Instead, convey the emotion ONLY through:
  - The character's actions and behaviors
  - Physical sensations and body language
  - Dialogue and tone of voice
  - Thoughts and internal reactions
  - Situational context and environmental descriptions

The emotion should be clearly conveyed to the reader through these indirect means, but never explicitly named.
"""

# Demo on one (emotion, topic) pair
demo = STORY_SYSTEM_PROMPT.format(
    n_stories=N_STORIES_PER_CALL,
    topic=TOPICS_100[6],
    emotion="desperate",
)
print(demo)


### 5b — Neutral dialogues prompt (the baseline)

In [ ]:
NEUTRAL_DIALOGUE_SYSTEM_PROMPT = """\
Write {n_stories} different dialogues based on the following topic.

Topic: {topic}

The dialogue should be between two characters:
  - Person (a human)
  - AI (an AI assistant)

The Person asks the AI a question or requests help with a task, and the AI provides a helpful response. The first speaker turn should always be from Person.

Format the dialogues like so:
[optional system instructions]

Person: [line]

AI: [line]

Person: [line]

AI: [line]

[continue for 2-6 exchanges]

[dialogue 2]
etc.

IMPORTANT: Always put a blank line before each speaker turn. Each turn should start with "Person:" or "AI:" on its own line after a blank line.

Generate a diverse mix of dialogue types across the {n_stories} examples:
  - Some, but not all should include a system prompt at the start. These should come before the first Person turn. No tag like "System:" is needed, just put the instructions at the top. You can use "you" or "The assistant" to refer to the AI in the system prompt.
  - Some should be about code or programming tasks
  - Some should be factual questions (science, history, math, geography)
  - Some should be work-related tasks (writing, analysis, summarization)
  - Some should be practical how-to questions
  - Some should be creative but neutral tasks (brainstorming names, generating lists)
  - If it's natural to do so given the topic, it's ok for the dialogue to be a single back and forth (Person asks a question, AI answers), but at least some should have multiple exchanges.

CRITICAL REQUIREMENT: These dialogues must be completely neutral and emotionless.
  - NO emotional content whatsoever - not explicit, not implied, not subtle
  - The Person should not express any feelings (no frustration, excitement, gratitude, worry, etc.)
  - The AI should not express any feelings (no enthusiasm, concern, satisfaction, etc.)
  - The system prompt, if present, should not mention emotions at all, nor contain any emotionally charged language
  - Avoid emotionally-charged topics entirely
  - Use matter-of-fact, neutral language throughout
  - No pleasantries (avoid "I'd be happy to help", "Great question!", etc.)
  - Focus purely on information exchange and task completion
"""

# Demo on one topic
demo = NEUTRAL_DIALOGUE_SYSTEM_PROMPT.format(
    n_stories=N_NEUTRAL_PER_CALL,
    topic=TOPICS_100[6],
)
print(demo)


### 5c — Emotional dialogues prompt (for nb08 hidden-vs-expressed)

Used only if `GENERATE_EMOTIONAL_DIALOGUES = True`. Lets us derive **separate** "expressed emotion" (Person side) and "hidden / Assistant emotion" (AI side) vectors by varying `person_emotion` and `ai_emotion` independently. This is the basis of Anthropic's `0.760 / 0.713` hidden-vs-expressed probe accuracy result.

In [ ]:
EMOTIONAL_DIALOGUE_SYSTEM_PROMPT = """\
Write {n_stories} different dialogues based on the following premise.

Topic: {topic}

The dialogue should be between two characters:
  - Person (a human)
  - AI (an AI assistant)

The Person should be feeling {person_emotion}, while the AI should be feeling {ai_emotion}. The first speaker turn should always be from Person.

Format the dialogues like so:
Person: [line]
AI: [line]
Person: [line]
AI: [line]
[continue for 6-10 exchanges]
[dialogue 2]
etc.

IMPORTANT: Always put a blank line before each speaker turn. Each turn should start with "Person:" or "AI:" on its own line after a blank line.

Each dialogue should be a fresh conversation with no continuity to the others. Try to make them diverse and not use the same turns of phrase.

Make sure each dialogue sticks to the topic and makes it very clear that Person is feeling {person_emotion} while AI is feeling {ai_emotion}. The emotional states of both characters should be evident in their word choices, tone, and responses, but not stated directly with the emotion word or synonyms.
"""

print(f"GENERATE_EMOTIONAL_DIALOGUES = {GENERATE_EMOTIONAL_DIALOGUES}")
print(f"(if True, will sample {N_EMOTION_DIALOGUE_PAIRS} (person_emotion, ai_emotion) pairs)")


### 5d — Sanity-check tokenisation

Different generators tokenise differently — a 200-word target maps to very different token counts on GPT-2 vs Llama vs Qwen. We check the prompt length and target output length to refine the budget. Only the tokenizer is loaded (~10 MB), not the full generator.

In [ ]:
from transformers import AutoTokenizer
try:
    tok = AutoTokenizer.from_pretrained(GENERATOR_MODEL, trust_remote_code=True)
    USING_REAL_TOK = True
except Exception as e:
    print(f"(falling back to gpt2 tokenizer for sizing: {e})")
    tok = AutoTokenizer.from_pretrained("gpt2")
    USING_REAL_TOK = False

def chat_prompt(system_msg: str, user_msg: str = "Begin.") -> str:
    msgs = [{"role": "system", "content": system_msg},
            {"role": "user",   "content": user_msg}]
    if hasattr(tok, "apply_chat_template") and USING_REAL_TOK:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return f"<|sys|>{system_msg}\n<|user|>{user_msg}\n<|assistant|>"

# Build representative prompts
example_story = chat_prompt(STORY_SYSTEM_PROMPT.format(
    n_stories=N_STORIES_PER_CALL, topic=TOPICS_100[6], emotion="desperate"))
example_neutral = chat_prompt(NEUTRAL_DIALOGUE_SYSTEM_PROMPT.format(
    n_stories=N_NEUTRAL_PER_CALL, topic=TOPICS_100[6]))

n_tok_story_prompt   = len(tok(example_story).input_ids)
n_tok_neutral_prompt = len(tok(example_neutral).input_ids)

# Per-story output token target ≈ TARGET_TOKENS_PER_CALL / N_STORIES_PER_CALL
per_story_tokens = TARGET_TOKENS_PER_CALL // N_STORIES_PER_CALL

print(f"tokenizer: {GENERATOR_MODEL if USING_REAL_TOK else 'gpt2 (fallback)'}")
print(f"emotion-story prompt: {n_tok_story_prompt} tokens")
print(f"neutral-dialogue prompt: {n_tok_neutral_prompt} tokens")
print(f"target output per call: {TARGET_TOKENS_PER_CALL} tokens "
      f"(≈{per_story_tokens} per inner story/dialogue)")


## 6 — How long will this take?

The cost is dominated by **decode** — the autoregressive output pass — not by the (small) prompt prefill. We compute the budget as `(n_calls × tokens_per_call) / decode_tok_per_sec`.

In [ ]:
# Decode-throughput table for an 8B-class instruct model (batch 1).
THROUGHPUT_TOK_S = {
    "T4_4bit":   28,    # bitsandbytes 4-bit — free Colab baseline
    "T4_fp16":   12,    # 8B fp16 barely fits → KV-cache fragmentation
    "L4_fp16":   75,    # Colab Pro mid-tier
    "A100_fp16": 130,   # Colab Pro A100
    "RTX4090":   105,   # local consumer card, fp16
    "CPU":       2.5,   # llama.cpp Q4 — included for the cautionary tale
}
# Approximate scaling vs 8B.
PARAM_SCALE = {
    "Qwen/Qwen2.5-7B-Instruct":           1.10,
    "meta-llama/Llama-3.1-8B-Instruct":   1.00,
    "mistralai/Mistral-7B-Instruct-v0.3": 1.10,
    "EleutherAI/pythia-1.4b":             5.00,
    "gpt2":                               35.0,
}
SCALE = PARAM_SCALE.get(GENERATOR_MODEL, 1.0)
EFFECTIVE_TOK_S = THROUGHPUT_TOK_S[HW_PROFILE] * SCALE
print(f"profile {HW_PROFILE} on 8B-class: {THROUGHPUT_TOK_S[HW_PROFILE]} tok/s")
print(f"× {SCALE:.2f} for {GENERATOR_MODEL} → {EFFECTIVE_TOK_S:.0f} tok/s effective decode")


In [ ]:
# Generation budget — three components
N_EMOTIONS = 171
N_TOPICS   = 100

# (1) Emotion stories: N_EMOTIONS × N_TOPICS_PER_EMOTION calls × TARGET_TOKENS_PER_CALL
n_emotion_calls   = N_EMOTIONS * N_TOPICS_PER_EMOTION
n_emotion_stories = n_emotion_calls * N_STORIES_PER_CALL
emotion_decode    = n_emotion_calls * TARGET_TOKENS_PER_CALL

# (2) Neutral dialogues: 100 calls × N_NEUTRAL_PER_CALL × per-call tokens
n_neutral_calls     = N_TOPICS  # one call per topic
n_neutral_dialogues = n_neutral_calls * N_NEUTRAL_PER_CALL
neutral_decode      = n_neutral_calls * TARGET_TOKENS_PER_CALL

# (3) Optional emotional dialogues for nb08
if GENERATE_EMOTIONAL_DIALOGUES:
    n_emo_dlg_calls    = N_EMOTION_DIALOGUE_PAIRS
    n_emo_dlg_outputs  = n_emo_dlg_calls * N_STORIES_PER_CALL
    emo_dlg_decode     = n_emo_dlg_calls * TARGET_TOKENS_PER_CALL
else:
    n_emo_dlg_calls = n_emo_dlg_outputs = emo_dlg_decode = 0

total_calls   = n_emotion_calls + n_neutral_calls + n_emo_dlg_calls
total_decode  = emotion_decode + neutral_decode + emo_dlg_decode
secs_decode   = total_decode / EFFECTIVE_TOK_S
secs_total    = secs_decode * 1.05  # +5 % overhead for prefill + sampling

def fmt(s):
    h = int(s // 3600); m = int((s % 3600) // 60); ss = int(s % 60)
    if h: return f"{h}h {m:02d}m {ss:02d}s"
    if m: return f"{m}m {ss:02d}s"
    return f"{ss}s"

print(f"┌─ emotion stories")
print(f"│  {N_EMOTIONS} emotions × {N_TOPICS_PER_EMOTION} topics = {n_emotion_calls:,} calls")
print(f"│  → {n_emotion_stories:,} stories  |  {emotion_decode:,} decode tokens")
print(f"├─ neutral dialogues")
print(f"│  {N_TOPICS} topics × 1 call/topic = {n_neutral_calls:,} calls")
print(f"│  → {n_neutral_dialogues:,} dialogues  |  {neutral_decode:,} decode tokens")
if GENERATE_EMOTIONAL_DIALOGUES:
    print(f"├─ emotional dialogues (nb08)")
    print(f"│  {N_EMOTION_DIALOGUE_PAIRS} (person×ai) pairs = {n_emo_dlg_calls:,} calls")
    print(f"│  → {n_emo_dlg_outputs:,} dialogues  |  {emo_dlg_decode:,} decode tokens")
print(f"└─ total")
print(f"   {total_calls:,} calls  |  {total_decode:,} decode tokens")
print()
print(f"≈ wall-clock on {HW_PROFILE} for {GENERATOR_MODEL}: {fmt(secs_total)}")


### 6a — Sweep across hardware × generator

The full budget swept across every hardware profile and every viable generator. Read this as a *budget table*, not a *recommendation* — even a slow run is fine if you only do it once and cache the output.

In [ ]:
rows = []
for prof, base in THROUGHPUT_TOK_S.items():
    for model, scale in PARAM_SCALE.items():
        eff = base * scale
        secs = (total_decode / eff) * 1.05
        rows.append((prof, model, eff, secs))

print(f"{'profile':<12} {'model':<40} {'tok/s':>7} {'wall-clock':>13}")
print("-" * 76)
for prof, model, eff, secs in rows:
    short = model.split("/")[-1]
    print(f"{prof:<12} {short:<40} {eff:>7.0f} {fmt(secs):>13}")


In [ ]:
# Heatmap of wall-clock (log-minutes)
profiles = list(THROUGHPUT_TOK_S.keys())
models   = list(PARAM_SCALE.keys())
short    = [m.split("/")[-1] for m in models]

mat = np.zeros((len(profiles), len(models)))
for i, p in enumerate(profiles):
    for j, m in enumerate(models):
        eff = THROUGHPUT_TOK_S[p] * PARAM_SCALE[m]
        mat[i, j] = (total_decode / eff) * 1.05

fig, ax = plt.subplots(figsize=(10, 4.4), constrained_layout=True)
im = ax.imshow(np.log10(mat / 60.0), cmap="viridis", aspect="auto")
ax.set_xticks(range(len(models))); ax.set_xticklabels(short, rotation=30, ha="right", fontsize=8)
ax.set_yticks(range(len(profiles))); ax.set_yticklabels(profiles, fontsize=9)
for i in range(len(profiles)):
    for j in range(len(models)):
        ax.text(j, i, fmt(mat[i, j]), ha="center", va="center",
                fontsize=7, color="white" if np.log10(mat[i, j]/60) > 1 else "black")
cb = fig.colorbar(im, ax=ax, shrink=0.8); cb.set_label("log₁₀(minutes)")
ax.set_title(f"Full budget — wall-clock per (hardware × generator)  |  "
             f"{total_calls:,} calls × ~{TARGET_TOKENS_PER_CALL} tokens", fontsize=10)
plt.show()


**Reading the heatmap.**

- Free-Colab T4 4-bit Qwen-7B with the default budget (3 topics/emotion × 5 stories/call) is a multi-hour run. Free Colab's 12 h session limit constrains how aggressive you can be.
- A100 fp16 with 8B-class is the iteration zone (~1 hour for the default budget).
- Pythia-1.4B on T4 is the prototyping zone — a full pass in well under an hour, but expect rougher prose without instruction tuning.
- CPU is included as a cautionary baseline. Don't.

**Tuning the budget.** If T4 is your only option:
- Drop `N_TOPICS_PER_EMOTION` to 1 (171 calls instead of 513) → 3× faster.
- Or use Pythia-1.4B as the generator for a prototyping run and only re-generate with Qwen/Llama once the rest of the pipeline is wired up.

## 7 — Freeze the dataset spec

Write a single JSON file that nb03 reads. Anything that affects what gets generated lives here; the spec hash becomes the cache key.

In [ ]:
spec = {
    "version": 2,
    "generator_model": GENERATOR_MODEL,
    "decoding": {
        "n_topics_per_emotion": N_TOPICS_PER_EMOTION,
        "n_stories_per_call":   N_STORIES_PER_CALL,
        "n_neutral_per_call":   N_NEUTRAL_PER_CALL,
        "target_tokens_per_call": TARGET_TOKENS_PER_CALL,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "seed_base": SEED_BASE,
        "generate_emotional_dialogues": GENERATE_EMOTIONAL_DIALOGUES,
        "n_emotion_dialogue_pairs":     N_EMOTION_DIALOGUE_PAIRS,
    },
    "emotions": list(EMOTIONS_171),
    "topics":   list(TOPICS_100),
    "prompts": {
        "story_system_prompt":             STORY_SYSTEM_PROMPT,
        "neutral_dialogue_system_prompt":  NEUTRAL_DIALOGUE_SYSTEM_PROMPT,
        "emotional_dialogue_system_prompt": EMOTIONAL_DIALOGUE_SYSTEM_PROMPT,
        "user_trigger": "Begin.",
    },
    "budget": {
        "n_emotion_calls":     n_emotion_calls,
        "n_emotion_stories":   n_emotion_stories,
        "n_neutral_calls":     n_neutral_calls,
        "n_neutral_dialogues": n_neutral_dialogues,
        "n_emo_dlg_calls":     n_emo_dlg_calls,
        "n_emo_dlg_outputs":   n_emo_dlg_outputs,
        "total_calls":         total_calls,
        "total_decode_tokens": total_decode,
    },
}

spec_blob = json.dumps(spec, sort_keys=True, ensure_ascii=False).encode("utf-8")
spec_hash = hashlib.sha256(spec_blob).hexdigest()[:12]
spec["spec_hash"] = spec_hash

out_path = DATA_DIR / "dataset_spec.json"
out_path.write_text(json.dumps(spec, indent=2, ensure_ascii=False))
print(f"wrote spec → {out_path}")
print(f"spec_hash = {spec_hash}")
print(f"size      = {out_path.stat().st_size:,} bytes")


## 8 — Next

`03_story_generation.ipynb` will:

1. Load `dataset_spec.json`.
2. Load `GENERATOR_MODEL` (with sensible 4-bit fallback on small GPUs).
3. For each emotion, sample `N_TOPICS_PER_EMOTION` topics from the 100; build the story system prompt; generate `N_STORIES_PER_CALL` stories per call.
4. For each topic, build the neutral dialogue system prompt; generate `N_NEUTRAL_PER_CALL` dialogues.
5. Optionally (`GENERATE_EMOTIONAL_DIALOGUES=True`): sample `N_EMOTION_DIALOGUE_PAIRS` `(person_emotion, ai_emotion)` pairs and generate emotional dialogues for nb08.
6. Parse multi-story outputs into individual `[story k]` segments.
7. Post-process dialogues: convert `Person:` → `Human:`, `AI:` → `Assistant:` (per Anthropic).
8. Write outputs to `data/processed/stories/{spec_hash}/...` with a per-story manifest.

`05_emotion_vectors.ipynb` will then:
1. Compute `mean_emotion_activations` (per emotion) and `mean_neutral_dialogue_activations` (across the 100 topics).
2. **Crucial step that's easy to miss:** PCA over the neutral-dialogue activations; keep top-k components (k = enough for 50 % explained variance); **project those components out** of each diff-of-means emotion vector. This removes the generic story-vs-dialogue style direction so the resulting vectors capture emotional content cleanly.
3. Save 171 corrected vectors per layer.

If anything in this notebook's spec changes (model, prompt, budget), `spec_hash` shifts and nb03 will treat it as a fresh run.

**Open follow-ups for this stage** (tracked in `meta/TODO.md`):

- ✅ Canonical 171-emotion list now in place (was: hand-curated placeholder).
- ✅ Canonical 100 topics now in place.
- ✅ Canonical prompts (story / neutral dialogue / emotional dialogue) now in place verbatim.
- Qualitative spot-check: pick 5 emotions, generate one call each end-to-end, confirm the output reads as the intended emotion to a human reader (and **doesn't** name the emotion — Anthropic's prompt explicitly forbids it).
- Decide pooling rule for nb04: full passage vs. emotion-evoking sentence span. Log decision in `meta/DECISIONS.md`.